In [0]:
storage_key = dbutils.secrets.get(scope="kv-finbank", key="storage-account-key")
spark.conf.set("fs.azure.account.key.stfinbankdevfbcq2026.dfs.core.windows.net",storage_key)

In [0]:
GOLD_BASE = "abfss://gold@stfinbankdevfbcq2026.dfs.core.windows.net"
CALIDAD_PATH = f"{GOLD_BASE}/_calidad/reporte_calidad"
 
resultados = []  
 
 
def registrar_resultado(nombre_prueba, tipo, aprobo, detalle):
    resultado = {
        "nombre_prueba": nombre_prueba,
        "tipo": tipo,
        "resultado": "PASS" if aprobo else "FAIL",
        "detalle": detalle,
    }
    resultados.append(resultado)
    simbolo = "OK" if aprobo else "FALLO"
    print(f"[{simbolo}] {nombre_prueba}: {detalle}")

In [0]:
def prueba_1_unicidad_clientes():
    df = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
    total = df.count()
    duplicados = df.groupBy("id_cli").count().filter("count > 1").count()
 
    aprobo = duplicados == 0
    detalle = f"{total} clientes, {duplicados} id_cli duplicados encontrados"
    registrar_resultado("Unicidad de id_cli en dim_clientes", "Unicidad", aprobo, detalle)
 
 
prueba_1_unicidad_clientes()

In [0]:
def prueba_2_completitud_transacciones():
    from pyspark.sql.functions import col
 
    df = spark.read.format("delta").load(f"{GOLD_BASE}/fact_transacciones")
    total = df.count()
 
    campos_obligatorios = ["id_mov", "id_cli", "vr_mov"]
    nulos_totales = 0
    detalle_por_campo = []
    for campo in campos_obligatorios:
        n = df.filter(col(campo).isNull()).count()
        nulos_totales += n
        detalle_por_campo.append(f"{campo}={n}")
 
    aprobo = nulos_totales == 0
    detalle = f"{total} filas revisadas, nulos por campo: {', '.join(detalle_por_campo)}"
    registrar_resultado("Completitud de campos obligatorios en fact_transacciones", "Completitud", aprobo, detalle)
 
 
prueba_2_completitud_transacciones()

In [0]:
def prueba_3_integridad_referencial():
    df_trans = spark.read.format("delta").load(f"{GOLD_BASE}/fact_transacciones")
    df_clientes = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
 
    ids_validos = df_clientes.select("id_cli").distinct()
    huerfanos = df_trans.join(
        ids_validos, df_trans["id_cli"] == ids_validos["id_cli"], "left_anti"
    ).count()
 
    aprobo = huerfanos == 0
    detalle = f"{huerfanos} transacciones con id_cli que no existe en dim_clientes, re-verificacion tras Silver)"
    registrar_resultado("Integridad referencial fact_transacciones A dim_clientes", "Integridad referencial", aprobo, detalle)
 
 
prueba_3_integridad_referencial()

In [0]:
def prueba_4_validez_bucket_mora():
    df = spark.read.format("delta").load(f"{GOLD_BASE}/fact_cartera")
 
    valores_esperados = {"Al dia", "Rango 1", "Rango 2", "Rango 3", "Deteriorado"}
    valores_reales = set(row["bucket_mora"] for row in df.select("bucket_mora").distinct().collect())
    valores_invalidos = valores_reales - valores_esperados
 
    aprobo = len(valores_invalidos) == 0
    detalle = f"valores encontrados: {valores_reales}. invalidos: {valores_invalidos if valores_invalidos else 'ninguno'}"
    registrar_resultado("validez de categorias en bucket_mora", "Validez / Rangos", aprobo, detalle)
 
 
prueba_4_validez_bucket_mora()

In [0]:
def prueba_5_consistencia_texto():
    from pyspark.sql.functions import trim, upper, col
 
    df = spark.read.format("delta").load(f"{GOLD_BASE}/dim_clientes")
 
    problemas = []
    for campo in ["ciudad_res", "segmento_legible"]:
        valores_originales = df.select(campo).distinct().count()
        valores_normalizados = (
            df.select(trim(upper(col(campo))).alias(campo)).distinct().count()
        )
        if valores_originales != valores_normalizados:
            problemas.append(
                f"{campo}: {valores_originales} valores distintos originales vs "
                f"{valores_normalizados} tras normalizar (posible inconsistencia de formato)"
            )
 
    aprobo = len(problemas) == 0
    detalle = "; ".join(problemas) if problemas else "ciudad_res y segmento_legible sin inconsistencias de formato detectadas"
    registrar_resultado("Consistencia de formato en columnas categoricas", "Consistencia", aprobo, detalle)
 
 
prueba_5_consistencia_texto()

In [0]:
def guardar_reporte_calidad():
    from pyspark.sql.types import StructType, StructField, StringType
    from datetime import datetime, timezone
 
    schema_reporte = StructType([
        StructField("nombre_prueba", StringType(), False),
        StructField("tipo", StringType(), False),
        StructField("resultado", StringType(), False),
        StructField("detalle", StringType(), False),
    ])
 
    df_reporte = spark.createDataFrame(resultados, schema=schema_reporte)
 
    from pyspark.sql.functions import lit, current_timestamp
    df_reporte = df_reporte.withColumn("fecha_ejecucion", current_timestamp())
 
    df_reporte.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(CALIDAD_PATH)
 
    total_pass = sum(1 for r in resultados if r["resultado"] == "PASS")
    total_fail = sum(1 for r in resultados if r["resultado"] == "FAIL")
    print(f"\nREPORTE FINAL")
    print(f"Pruebas ejecutadas: {len(resultados)}")
    print(f"PASS: {total_pass}  |  FAIL: {total_fail}")
 
    return df_reporte
 
 
df_reporte_calidad = guardar_reporte_calidad()
display(df_reporte_calidad)